# 08 — 用 SDK 编写可查询、可解释的 Policy

本教程演示新的用户层 Policy API。你不需要手写 `PolicyAll`、`PolicyAny`、`PolicyOccurrence`、`SemanticPortAddress` 或 `SemanticAddressSpace`；它们仍然存在于应用层，但 SDK 将它们封装为带类型和所有权检查的草稿、occurrence 与 port handles。

本教程会完成：

1. 使用 `fg.policy(...).use(...).build(...)` 构造不可变 Policy；
2. 写 `people.age > 12`、`older.age > younger.age` 与 one-hop field navigation；
3. 用显式 `draft.all(...)` / `draft.any(...)` 保留嵌套逻辑拓扑；
4. 将同一批 port handles 用于 `fg.query(...).bind(...).select(...)`；
5. 得到 sealed Run、显式 row Explain、detached replay，以及三引擎 selected-row-set parity。

> **边界。** 这是一个声明式 Policy authoring surface，不是任意 Python 求值器。`and` / `or`、链式比较、隐式 entity equality、字符串 Policy 查找、Actions、multi-hop navigation 与任意字面量都不会被悄悄解释。

## 运行方法

从仓库根目录（或 `examples/`）以 `factpy` 环境启动 Jupyter。每个 cell 都使用真实 SDK 与真实 rule compiler。最后一个 portable cell 还要求本机有 Native、Soufflé 与 ProbLog，和项目测试环境一致。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / 'src').exists() and (_repo_root.parent / 'src').exists():
    _repo_root = _repo_root.parent
if str(_repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(_repo_root / 'src'))

from factgraph.application import build_resolved_rule, build_schema_index
from factgraph.application.protocol import (
    EntityRef,
    EntitySelector,
    SemanticRulePort,
    entity_identity,
    field_endpoint,
)
from factgraph.application.schema_runtime import entity_info, field_predicate, resolve_selector
from factgraph.core.evidence.write_protocol import set_field
from factgraph.core.rules.where_ast import PredAtom, Var
from factgraph.sdk import (
    Entity,
    ExplainTargetV1,
    Field,
    GoalPlanRunV1,
    Identity,
    PolicyAuthoringError,
    SDKStore,
    native_deterministic_profile_v1,
    portable_deterministic_profile_v1,
)


## 示例数据与已解析 Rule

下面的 fixture 只负责准备 schema、事实和一个**已解析 Rule asset**。真实业务中，Rule 可以来自你们的受控 authoring/发布流程；从 `fg.policy(...)` 开始的 cell 才是本教程要展示的用户层 Policy API。

In [ ]:
class Person(Entity):
    employee_id: str = Identity()
    age: int = Field()
    score: int = Field()


def person_values_rule(graph: SDKStore):
    index = build_schema_index(graph.schema_ir)
    person, age, score = Var('$person'), Var('$age'), Var('$score')
    return build_resolved_rule(
        id='person_values',
        version='1',
        when=(
            PredAtom('Person:exists', [person]),
            PredAtom('person:age', [person, age]),
            PredAtom('person:score', [person, score]),
        ),
        ports={
            'person': SemanticRulePort(person, entity_identity('Person')),
            'age': SemanticRulePort(age, field_endpoint('Person', 'age')),
            'score': SemanticRulePort(score, field_endpoint('Person', 'score')),
        },
        schema_index=index,
    )


def seed_person(graph: SDKStore, employee_id: str, *, age: int, score: int) -> str:
    # Fixture-only ledger setup. Production code normally uses the SDK ingest/write path.
    index = build_schema_index(graph.schema_ir)
    ref = resolve_selector(
        EntitySelector(entity_type='Person', identity={'employee_id': employee_id}),
        index=index,
    )
    encoded = ref.encoded_ref or ''
    info = entity_info(index, 'Person')
    set_field(graph.ledger, info.exists_predicate_id, encoded, [])
    set_field(graph.ledger, info.identity_predicates['employee_id'].pred_id, encoded, [('string', employee_id)])
    set_field(graph.ledger, field_predicate(index, 'Person', 'age').pred_id, encoded, [('int', age)])
    set_field(graph.ledger, field_predicate(index, 'Person', 'score').pred_id, encoded, [('int', score)])
    return encoded


## 1. 从草稿到 Query target

`use(..., as_=...)` 给已解析 Rule 一个 Policy 内的 occurrence 名称。`people.person` 和 `people.age` 不是字符串路径，而是 schema-checked typed handles。`people.age > 12` 会封存为 `PolicyLiteral('int', 12)`，随后由既有 Policy compiler 降到同一份逻辑 IR。

In [ ]:
fg = SDKStore([Person])
alice_ref = seed_person(fg, 'alice', age=22, score=7)

adult = fg.policy('adult_review', version='1')
people = adult.use(person_values_rule(fg), as_='people')
adult_target = adult.build(adult.all(people, people.age > 12))

outcome = (
    fg.query(adult_target)
      .bind(people.person, EntityRef('Person', {'employee_id': 'alice'}))
      .select('age', people.age)
      .plan(profile=native_deterministic_profile_v1())
      .run()
)
assert isinstance(outcome, GoalPlanRunV1), outcome
run = outcome.run
assert run.effective.canonical_result.rows[0].values[0][1].value == 22
assert adult_target.policy.id == 'adult_review'
print('selected row:', run.effective.canonical_result.rows[0].values)


## 2. 同一 target 进入 sealed Run、Explain 与 replay

Query 不需要知道 Policy 是 Rule lift 还是多 occurrence Policy。它只接收已解析 target、typed bind 和 ordered select。Explain 必须明确指定一条 positive row；它从 sealed relation 重算 native inner evidence，再投影回 authored Policy topology。

In [ ]:
row = run.effective.canonical_result.rows[0]
assert row.anchor is not None
explanation = outcome.explain(
    ExplainTargetV1('effective', 'row', row.anchor.anchor_digest)
)
assert explanation.engine_evidence == 'native_detached_recomputed'
assert explanation.policy_projection is not None
assert explanation.proof_parity == 'not_claimed'
assert outcome.replay().status == 'matched'
print('policy projection:', explanation.policy_projection is not None)
print('proof parity:', explanation.proof_parity)


## 3. 比较、field navigation 与嵌套 `all` / `any`

显式 `all` / `any` 比 `&` / `|` 更适合 Policy：它避免 Python precedence 陷阱，并保留 Explain 所需的 authored nesting。每个 occurrence 在一个 Policy 中必须出现一次；若一个 Rule 要出现在多个分支，请为每次语义 occurrence 调用一次 `use(...)`。

`older.age > 12` 是 direct scalar compare。`older.person.field('age') > younger.person.field('age')` 是 one-hop same-entity field navigation compare；`.field(...)` 是无歧义主写法，`older.person.age` 仅是安全标识符的等价简写。

In [ ]:
ranked_fg = SDKStore([Person])
seed_person(ranked_fg, 'alice', age=30, score=7)
seed_person(ranked_fg, 'bob', age=20, score=4)
rule = person_values_rule(ranked_fg)

ranked = ranked_fg.policy('ranked_people', version='1')
older = ranked.use(rule, as_='older')
younger = ranked.use(rule, as_='younger')
left_gate = ranked.use(rule, as_='left_gate')
right_gate = ranked.use(rule, as_='right_gate')
ranked_target = ranked.build(
    ranked.all(
        older,
        younger,
        older.age > 12,
        older.person.field('age') > younger.person.field('age'),
        ranked.any(ranked.all(left_gate), ranked.all(right_gate)),
    )
)

compiled = (
    ranked_fg.query(ranked_target)
      .bind(older.person, EntityRef('Person', {'employee_id': 'alice'}))
      .select('age', older.age)
      .compile()
)
result = ranked_fg.eval.evaluate(compiled)
assert result[0].bindings['age']['value'] == 30
structure = compiled.target.compiled_policy.policy_structure
assert {'all', 'any', 'occurrence', 'compare'} <= {node.kind for node in structure.nodes}
print('authored node kinds:', sorted({node.kind for node in structure.nodes}))


## 4. Entity identity 必须显式

`==` 用于 scalar compare。实体 identity 的合一使用 `draft.same(...)`，这样它在 authored topology、lineage 与 Explain 中是一个具名的 `Unify` condition，而不是难以审计的 Python equality。

In [ ]:
same_person = fg.policy('same_person', version='1')
left = same_person.use(person_values_rule(fg), as_='left')
right = same_person.use(person_values_rule(fg), as_='right')
same_person_target = same_person.build(
    same_person.all(left, right, same_person.same(left.person, right.person))
)
assert same_person_target.policy.id == 'same_person'


## 5. Python 语法陷阱会失败，而不是改变逻辑

Policy handles 不能当作 bool 或 hash key。这样 `and` / `or`、条件表达式、链式比较以及把 handle 放进 set/dict 都不会产生一个看似合理、实际语义错误的 Policy。

In [ ]:
def show_rejection(label, thunk):
    try:
        thunk()
    except (PolicyAuthoringError, TypeError) as exc:
        print(f'{label}: {type(exc).__name__} -> {exc}')
    else:
        raise AssertionError(f'{label} should have been rejected')

show_rejection('Python and', lambda: (people.age > 12) and (people.age < 65))
show_rejection('chained comparison', lambda: 12 < people.age < 65)
show_rejection('implicit entity equality', lambda: left.person == right.person)
show_rejection('symbolic hash key', lambda: hash(people.age))

# Correct explicit form:
age_band = adult.all(people, people.age >= 12, people.age < 65)
assert age_band is not None


## 6. Portable profile：同一 Policy、同一 sealed relation、三引擎 row-set parity

`portable_deterministic_v1` 在 isolated materialized relation 上分别执行 Native、Soufflé 和 ProbLog，并比较 canonical selected-row set。它不是 proof/evidence byte parity；Explain 仍以 captured relation 上重算的 native inner evidence 为准。

In [ ]:
portable_outcome = (
    fg.query(adult_target)
      .bind(people.person, EntityRef('Person', {'employee_id': 'alice'}))
      .select('age', people.age)
      .plan(profile=portable_deterministic_profile_v1())
      .run()
)
assert isinstance(portable_outcome, GoalPlanRunV1), portable_outcome
portable_run = portable_outcome.run
assert tuple(frame.engine for frame in portable_run.effective.engine_results) == (
    'native', 'souffle', 'problog'
)
assert portable_run.effective.assessment.parity == 'equivalent'
portable_row = portable_run.effective.canonical_result.rows[0]
assert portable_row.anchor is not None
portable_explain = portable_outcome.explain(
    ExplainTargetV1('effective', 'row', portable_row.anchor.anchor_digest)
)
assert portable_explain.engine_evidence == 'native_detached_recomputed'
assert portable_explain.proof_parity == 'not_claimed'
print('engines:', [frame.engine for frame in portable_run.effective.engine_results])
print('row-set parity:', portable_run.effective.assessment.parity)


## 7. 这如何服务 AgentPlan

对 Agent 而言，目标不是生成任意 Python，而是生成受限、声明式的 Policy/Query 意图：已发布的 Rule asset、occurrence aliases、typed port references、`all`/`any` tree、比较、bind、select、Scenario、expectations 与 profile pins。AgentPlan 编译器可以把这种结构化意图映射到本教程中的同一 Policy/Query contract。

这正是“大一统 Query”在正确边界内的含义：Rule、Policy、受限 Provider 和 Scenario 参与同一个调用、组合、Run、审计接口；它们不因此变成同一种底层逻辑，也不会获得未声明的 Action、全局否定、任意外部 I/O 或跨引擎 proof parity。

### 当前明确支持 / 明确不支持

| 支持 | 当前明确拒绝或留给后续设计 |
|---|---|
| resolved Rule / SDK-authored Policy Query target | bare string/registry lookup、bare provider Query |
| nested `all` / `any`、explicit entity `same` | Python `and` / `or` / `not`、implicit entity equality |
| direct scalar / one-hop field navigation compare | multi-hop/multi-value navigation、relationship logic |
| `int` / `time` signed-int64 literals | strings, bools, floats, UUIDs, bytes, `None` literals |
| sealed Run, explicit positive-row Explain, replay | zero-row negative proof / implicit first-row Explain |
| Native/Soufflé/ProbLog selected-row-set parity in the portable fragment | proof/evidence parity, adapter fallback, arbitrary engine code |
